# 🌊 AquaSentinel AI: YOLO-Seg Side-Scan Sonar Training Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RaghavKacker/Aqua-Sentinel/blob/main/notebooks/AquaSentinel_YOLO_Seg_Colab_Training.ipynb)

This notebook trains an **Ultralytics YOLO-Seg (Instance Segmentation)** model for underwater marine debris and anomaly detection from **Side-Scan Sonar (SSS)** imagery.

### Target Classes:
* `0: crab_pot` (Ghost gear / crab traps)
* `1: ghost_gear` (Entangled nets & ropes)
* `2: mine_cylinder` (Cylindrical objects & AUV targets)
* `3: debris_anomaly` (Structural marine anomalies)

---

## 1. Verify GPU Acceleration
Ensure your Google Colab runtime is configured to use a GPU (**Runtime** → **Change runtime type** → **T4 GPU**).

In [ ]:
!nvidia-smi

## 2. Install Ultralytics and Required Libraries

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml matplotlib

import torch
import ultralytics
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
ultralytics.checks()

## 3. Clone Repository or Set Up Workspace
If running in Colab, clone the AquaSentinel project repository to access scripts and configurations.

In [ ]:
import os
if not os.path.exists('Aqua-Sentinel'):
    !git clone https://github.com/RaghavKacker/Aqua-Sentinel.git
    %cd Aqua-Sentinel
else:
    %cd Aqua-Sentinel

!pwd

## 4. Prepare Dataset (GhostVision + SSS-Mine / NOMBO or Demo Generator)

You can either:
1. **Use Real Datasets**: Place your GhostVision and SSS-Mine datasets in `data/`.
2. **Use Synthetic Demo Generator**: Generate realistic SSS patches immediately without downloading large archives.

In [ ]:
# Generate a balanced, mission-split SSS dataset (300 samples across 6 surveys)
!python ml/prepare_dataset.py --demo --samples 300 --output_dir ./data/unified_yolo_seg

# Check generated dataset config
!cat ./data/unified_yolo_seg/dataset.yaml

## 5. Visualize Sample Training Images & Labels

In [ ]:
import cv2
import glob
import matplotlib.pyplot as plt

sample_images = glob.glob('./data/unified_yolo_seg/images/train/*.png')[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, img_path in enumerate(sample_images):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img_rgb)
    axes[i].set_title(os.path.basename(img_path), fontsize=9)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 6. Train YOLO-Seg Model
We use `yolo11n-seg.pt` (Nano segmentation model) for lightweight, high-speed subsea inference.

In [ ]:
from ultralytics import YOLO

# Initialize YOLO-Seg model
model = YOLO('yolo11n-seg.pt')

# Train model on GPU
results = model.train(
    data='./data/unified_yolo_seg/dataset.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    project='./runs/train',
    name='aquasentinel_seg',
    exist_ok=True,
    plots=True,
    save=True,
    mosaic=0.5,
    fliplr=0.5
)

# Display exact output path
if hasattr(model, 'trainer') and hasattr(model.trainer, 'best'):
    print(f'\n[SUCCESS] Training Complete! Best weights: {model.trainer.best}')


## 7. Evaluate Model on Unseen Test Mission

In [ ]:
# Validate on isolated test split (unseen survey)
metrics = model.val(
    data='./data/unified_yolo_seg/dataset.yaml',
    split='test',
    imgsz=640,
    plots=True
)

print('=' * 40)
print(f'Box mAP50:     {metrics.box.map50:.4f}')
print(f'Box mAP50-95:  {metrics.box.map:.4f}')
if hasattr(metrics, 'seg'):
    print(f'Mask mAP50:    {metrics.seg.map50:.4f}')
    print(f'Mask mAP50-95: {metrics.seg.map:.4f}')
print('=' * 40)

## 8. Run Inference & Display Segmentation Masks

In [ ]:
test_images = glob.glob('./data/unified_yolo_seg/images/test/*.png')[:3]
for test_img in test_images:
    res = model.predict(source=test_img, conf=0.3, save=True)

# Display predictions
from IPython.display import Image, display
pred_files = glob.glob('./runs/segment/predict*/*.jpg') + glob.glob('./runs/segment/predict*/*.png')
if pred_files:
    display(Image(filename=pred_files[-1]))

## 9. Export & Download best.pt for Local Deployment
Download `best.pt` to place inside your local AquaSentinel `models/` directory.

In [ ]:
import os
import glob
from google.colab import files

# Dynamically search for best.pt across all possible Ultralytics output directories
found_weights = glob.glob('runs/**/weights/best.pt', recursive=True) + glob.glob('**/best.pt', recursive=True)
found_weights = list(dict.fromkeys([w for w in found_weights if os.path.exists(w) and os.path.getsize(w) > 10000]))

if found_weights:
    target_path = found_weights[-1]  # Select the latest trained weights
    size_mb = os.path.getsize(target_path) / (1024 * 1024)
    print(f'[FOUND] Best weights located at: {target_path} ({size_mb:.2f} MB)')
    print('Starting automatic download to your computer...')
    files.download(target_path)
    print('\nDownload complete! Place this file into: AquaSentinel/models/best.pt')
else:
    print('[INFO] No trained weights file (best.pt) found yet.')
    print('\nPlease check:')
    print('1. Did you run Cell 6 (Train YOLO-Seg Model)?')
    print('2. Is Cell 6 still running? (Wait until all epochs finish and it says "Training Complete!")')
    print('3. If Cell 6 stopped with an error, check the output of Cell 6.')
